# 02 - Feature Engineering

このノートブックでは、ドメイン知識に基づいた特徴生成を行います。

## Contents
1. データ読み込み
2. ドメイン特徴の生成
3. 統計特徴の生成
4. カテゴリ変数のエンコーディング
5. 欠損値処理
6. 特徴量スケーリング
7. 最終データセット保存

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Paths
TRAIN_PATH = 'data/train.csv'
TEST_PATH = 'data/test.csv'
PROCESSED_DIR = Path('data/processed')
PROCESSED_DIR.mkdir(exist_ok=True)

print('Imports successful!')

## 1. データ読み込み

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print(f'Train shape: {train_df.shape}')
print(f'Test shape: {test_df.shape}')

# Separate target and features
target_col = 'Irrigation_Need'
id_col = 'id'

y_train = train_df[target_col].copy()
X_train = train_df.drop([target_col, id_col], axis=1).copy()
X_test = test_df.drop([id_col], axis=1).copy()

print(f'\nTrain features shape: {X_train.shape}')
print(f'Test features shape: {X_test.shape}')
print(f'Target shape: {y_train.shape}')

## 2. ドメイン特徴の生成

In [ ]:
def engineer_features(df, prefix=''):
    """
    Generate domain-aware features for irrigation prediction
    """
    df = df.copy()
    
    # 1. Soil-Weather Interactions
    if 'Soil_pH' in df.columns and 'Temperature_C' in df.columns:
        df[f'{prefix}pH_x_Temp'] = df['Soil_pH'] * df['Temperature_C']
    
    if 'Soil_Moisture' in df.columns and 'Rainfall_mm' in df.columns:
        df[f'{prefix}Moisture_x_Rainfall'] = df['Soil_Moisture'] * df['Rainfall_mm']
    
    # 2. Water Balance Features
    if 'Soil_Moisture' in df.columns and 'Rainfall_mm' in df.columns:
        df[f'{prefix}Moisture_to_Rainfall'] = df['Soil_Moisture'] / (df['Rainfall_mm'] + 1)
    
    if 'Previous_Irrigation_mm' in df.columns and 'Rainfall_mm' in df.columns:
        df[f'{prefix}Total_Water_Input'] = df['Previous_Irrigation_mm'] + df['Rainfall_mm']
    
    # 3. Irrigation Efficiency
    if 'Previous_Irrigation_mm' in df.columns and 'Field_Area_hectare' in df.columns:
        df[f'{prefix}Irrigation_per_Area'] = df['Previous_Irrigation_mm'] / (df['Field_Area_hectare'] + 0.01)
    
    # 4. Environmental Stress Index
    if 'Temperature_C' in df.columns and 'Humidity' in df.columns:
        df[f'{prefix}Heat_Stress'] = df['Temperature_C'] * (100 - df['Humidity']) / 100
    
    if 'Wind_Speed_kmh' in df.columns and 'Humidity' in df.columns:
        df[f'{prefix}Evaporation_Stress'] = df['Wind_Speed_kmh'] * (100 - df['Humidity']) / 100
    
    # 5. Soil Quality Metrics
    if 'Electrical_Conductivity' in df.columns and 'Organic_Carbon' in df.columns:
        df[f'{prefix}Soil_Quality'] = df['Organic_Carbon'] / (df['Electrical_Conductivity'] + 0.001)
    
    # 6. Light Intensity Feature
    if 'Sunlight_Hours' in df.columns:
        df[f'{prefix}Sunlight_Intensity'] = df['Sunlight_Hours']  # normalized if needed
    
    # 7. Field Information
    if 'Field_Area_hectare' in df.columns:
        df[f'{prefix}Field_Size_Category'] = pd.cut(df['Field_Area_hectare'], 
                                                      bins=[-np.inf, 1, 5, 10, np.inf], 
                                                      labels=['Small', 'Medium', 'Large', 'VeryLarge'])
    
    return df

# Apply feature engineering
X_train_eng = engineer_features(X_train, prefix='eng_')
X_test_eng = engineer_features(X_test, prefix='eng_')

print(f'\nEngineered train shape: {X_train_eng.shape}')
print(f'Engineered test shape: {X_test_eng.shape}')
print(f'\nNew features created: {X_train_eng.shape[1] - X_train.shape[1]}')

## 3. 統計特徴の生成

In [ ]:
def create_statistical_features(df, prefix=''):
    """
    Create statistical aggregations for numerical features
    """
    df = df.copy()
    
    numerical_cols = df.select_dtypes(include=[np.number]).columns
    
    # Mean of all numerical features
    df[f'{prefix}num_mean'] = df[numerical_cols].mean(axis=1)
    
    # Std of numerical features
    df[f'{prefix}num_std'] = df[numerical_cols].std(axis=1)
    
    # Max-Min range
    df[f'{prefix}num_range'] = df[numerical_cols].max(axis=1) - df[numerical_cols].min(axis=1)
    
    return df

X_train_eng = create_statistical_features(X_train_eng, prefix='stat_')
X_test_eng = create_statistical_features(X_test_eng, prefix='stat_')

print(f'Train shape after statistical features: {X_train_eng.shape}')

## 4. カテゴリ変数のエンコーディング

In [ ]:
# Identify categorical columns
categorical_cols = X_train_eng.select_dtypes(include=['object']).columns.tolist()
print(f'Categorical columns: {categorical_cols}')

# Label encode categorical variables
le_dict = {}
for col in categorical_cols:
    le = LabelEncoder()
    X_train_eng[col] = le.fit_transform(X_train_eng[col].astype(str))
    X_test_eng[col] = le.transform(X_test_eng[col].astype(str))
    le_dict[col] = le
    print(f'  {col}: {len(le.classes_)} unique values')

print('\n✓ Categorical encoding complete')

## 5. 欠損値処理

In [ ]:
# Check for missing values
print('Missing values in train:')
null_counts = X_train_eng.isnull().sum()
if null_counts.sum() > 0:
    print(null_counts[null_counts > 0])
else:
    print('No missing values!')

# Fill missing values
numerical_cols = X_train_eng.select_dtypes(include=[np.number]).columns
for col in numerical_cols:
    median_val = X_train_eng[col].median()
    X_train_eng[col].fillna(median_val, inplace=True)
    X_test_eng[col].fillna(median_val, inplace=True)

print('\n✓ Missing values filled')

## 6. 特徴量スケーリング（オプション）

In [ ]:
# Optional: Standardize numerical features
scaler = StandardScaler()
numerical_cols = X_train_eng.select_dtypes(include=[np.number]).columns

X_train_scaled = X_train_eng.copy()
X_test_scaled = X_test_eng.copy()

X_train_scaled[numerical_cols] = scaler.fit_transform(X_train_eng[numerical_cols])
X_test_scaled[numerical_cols] = scaler.transform(X_test_eng[numerical_cols])

print(f'Scaled train mean: {X_train_scaled[numerical_cols].mean().mean():.6f}')
print(f'Scaled train std: {X_train_scaled[numerical_cols].std().mean():.6f}')

## 7. 最終データセット保存

In [ ]:
# Prepare final datasets with target
train_final = X_train_eng.copy()
train_final[target_col] = y_train

# Prepare test dataset
test_final = X_test_eng.copy()

# Save
train_final.to_csv(PROCESSED_DIR / 'train_engineered.csv', index=False)
test_final.to_csv(PROCESSED_DIR / 'test_engineered.csv', index=False)

print(f'✓ Train engineered: {train_final.shape}')
print(f'  Saved to: data/processed/train_engineered.csv')
print(f'\n✓ Test engineered: {test_final.shape}')
print(f'  Saved to: data/processed/test_engineered.csv')

# Save scaled versions
train_scaled_final = X_train_scaled.copy()
train_scaled_final[target_col] = y_train

train_scaled_final.to_csv(PROCESSED_DIR / 'train_engineered_scaled.csv', index=False)
X_test_scaled.to_csv(PROCESSED_DIR / 'test_engineered_scaled.csv', index=False)

print(f'\n✓ Scaled versions also saved')
print('\n=== Feature Engineering Complete ===')